In [ ]:
# Step 11: Final Interactive AI Chatbot

# ## Objective
# Integrate the previously developed NLP, machine learning,
# sentiment analysis, emotion detection, feedback and fallback
# components into one interactive chatbot.

# ## Main Features
# - Intent recognition
# - Named entity extraction
# - Sentiment analysis
# - Emotion detection
# - Adaptive responses
# - Confidence-based fallback
# - User feedback collection
# - Interaction history
# - Analytics-ready data storage

# ## Interface
# Jupyter Notebook with Python and ipywidgets.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import json
import joblib
import spacy
import ipywidgets as widgets

from pathlib import Path
from datetime import datetime
from IPython.display import display, clear_output

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("Final chatbot libraries imported successfully.")

Final chatbot libraries imported successfully.


In [2]:
# Check the interactive notebook environment

import sys

print("Python version:", sys.version)
print("ipywidgets version:", widgets.__version__)

print("Interactive notebook environment is ready.")

Python version: 3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]
ipywidgets version: 8.1.9
Interactive notebook environment is ready.


In [16]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [17]:
from pathlib import Path
import joblib
import pandas as pd

PROJECT_ROOT = Path(r"D:\Dynamic_AI_Chatbot_Project")

MODEL_PATH = PROJECT_ROOT / "models" / "nlp" / "intent_logistic_regression.pkl"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "chatbot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEEDBACK_PATH = OUTPUT_DIR / "chatbot_feedback.csv"
HISTORY_PATH = OUTPUT_DIR / "chatbot_interaction_history.csv"

# Check that the trained NLP model exists
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"NLP model not found: {MODEL_PATH}"
    )

# Load the trained intent classification pipeline
intent_model = joblib.load(MODEL_PATH)

print("Project paths configured successfully.")
print("NLP intent model loaded successfully.")
print("Model location:", MODEL_PATH)

Project paths configured successfully.
NLP intent model loaded successfully.
Model location: D:\Dynamic_AI_Chatbot_Project\models\nlp\intent_logistic_regression.pkl


In [18]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialise sentiment analyser
sentiment_analyzer = SentimentIntensityAnalyzer()


# 1. Text preprocessing
def preprocess_text(message):
    text = str(message).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# 2. Sentiment analysis
def analyze_sentiment(message):
    scores = sentiment_analyzer.polarity_scores(message)
    compound = scores["compound"]

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    return {
        "label": label,
        "positive": scores["pos"],
        "negative": scores["neg"],
        "neutral": scores["neu"],
        "compound": compound
    }


# 3. Emotion detection
emotion_keywords = {
    "Happy": ["happy", "excited", "wonderful", "great", "glad"],
    "Angry": ["angry", "furious", "annoyed", "mad"],
    "Sad": ["sad", "unhappy", "disappointed", "upset"],
    "Frustrated": ["frustrated", "irritated", "problem", "issue"],
    "Worried": ["worried", "concerned", "afraid", "anxious"]
}

def detect_emotion(message):
    text = message.lower()

    for emotion, keywords in emotion_keywords.items():
        if any(keyword in text for keyword in keywords):
            return emotion

    return "Neutral"


# 4. Rule-based entity extraction
def extract_entities(message):
    entities = {}

    patterns = {
        "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "phone": r"\b(?:\+?\d[\d\s-]{8,}\d)\b",
        "date": r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{1,2}\s+[A-Za-z]+\s+\d{4}\b",
        "time": r"\b\d{1,2}(?::\d{2})?\s?(?:AM|PM|am|pm)\b",
        "amount": r"\b(?:Rs\.?|INR|₹)\s?\d+(?:,\d{3})*(?:\.\d{1,2})?\b"
    }

    for entity_type, pattern in patterns.items():
        matches = re.findall(pattern, message)
        if matches:
            entities[entity_type] = matches

    services = [
        "internet", "broadband", "mobile", "billing",
        "payment", "technical support", "connection"
    ]

    found_services = [
        service for service in services
        if service in message.lower()
    ]

    if found_services:
        entities["service"] = found_services

    return entities


print("NLP processing functions created successfully.")

NLP processing functions created successfully.


In [19]:
# Responses for the 12 trained intents
response_templates = {
    "greeting": "Hello! Welcome. How can I help you today?",

    "goodbye": "Goodbye! Thank you for chatting with me.",

    "thanks": "You're welcome! I'm happy to help.",

    "bot_identity": "I am an AI-powered customer support chatbot.",

    "help": (
        "I can help with service information, billing questions, "
        "technical support, complaints and general enquiries."
    ),

    "faq_hours": (
        "Our support hours are 9 AM to 6 PM, Monday to Saturday. "
        "Please confirm the actual business hours before deployment."
    ),

    "faq_services": (
        "I can assist with internet, mobile, billing, payments "
        "and technical support enquiries."
    ),

    "request_support": (
        "I can help you with that. Please describe the issue "
        "you are experiencing."
    ),

    "complaint": (
        "I'm sorry you are experiencing this problem. "
        "Please share more details so I can understand your complaint."
    ),

    "positive_feedback": (
        "Thank you for your positive feedback! "
        "We appreciate your support."
    ),

    "negative_feedback": (
        "I'm sorry the experience was not satisfactory. "
        "Your feedback is important to us."
    ),

    "request_human": (
        "I understand you would like to speak with a human agent. "
        "This notebook demo cannot transfer you to a live agent, "
        "but your request can be recorded."
    )
}


def generate_response(intent, emotion, sentiment):
    response = response_templates.get(
        intent,
        "I'm not completely sure I understood. Could you rephrase your question?"
    )

    # Adapt the response for negative emotions
    if emotion in ["Angry", "Sad", "Frustrated", "Worried"]:
        response = (
            "I understand this may be frustrating. "
            + response
        )

    elif sentiment == "Positive" and intent not in [
        "thanks", "positive_feedback"
    ]:
        response = response + " I'm glad to assist you!"

    return response


print("Response generation function created successfully.")

Response generation function created successfully.


In [20]:
def process_message(message, threshold=0.45):

    message = str(message).strip()

    if not message:
        return {
            "response": "Please enter a message.",
            "intent": "empty",
            "confidence": 0.0,
            "sentiment": "Neutral",
            "emotion": "Neutral",
            "entities": {}
        }

    # Preprocess the message
    clean_message = preprocess_text(message)

    # Predict intent and confidence
    probabilities = intent_model.predict_proba([clean_message])[0]

    best_index = np.argmax(probabilities)
    confidence = float(probabilities[best_index])
    predicted_intent = str(intent_model.classes_[best_index])

    # Apply fallback threshold
    if confidence < threshold:
        intent = "fallback"
        response = (
            "I'm not fully sure I understood your message. "
            "Could you please explain it in another way?"
        )
    else:
        intent = predicted_intent

        # Analyse sentiment and emotion
        sentiment_result = analyze_sentiment(message)
        emotion = detect_emotion(message)

        response = generate_response(
            intent,
            emotion,
            sentiment_result["label"]
        )

    # Extract entities and analyse message
    sentiment_result = analyze_sentiment(message)
    emotion = detect_emotion(message)
    entities = extract_entities(message)

    return {
        "response": response,
        "intent": intent,
        "predicted_intent": predicted_intent,
        "confidence": round(confidence, 4),
        "sentiment": sentiment_result["label"],
        "emotion": emotion,
        "entities": entities
    }


print("Complete message-processing function created successfully.")

Complete message-processing function created successfully.


In [21]:
# Test the chatbot engine

test_messages = [
    "Hello, how are you?",
    "I need help with my internet connection.",
    "I am very frustrated with the billing problem.",
    "Thank you for your support.",
    "I want to speak with a human agent.",
    "My email is test@example.com"
]

for message in test_messages:
    result = process_message(message)

    print("=" * 60)
    print("User:", message)
    print("Bot:", result["response"])
    print("Intent:", result["intent"])
    print("Confidence:", result["confidence"])
    print("Sentiment:", result["sentiment"])
    print("Emotion:", result["emotion"])
    print("Entities:", result["entities"])

User: Hello, how are you?
Bot: I'm not fully sure I understood your message. Could you please explain it in another way?
Intent: fallback
Confidence: 0.2227
Sentiment: Neutral
Emotion: Neutral
Entities: {}
User: I need help with my internet connection.
Bot: I'm not fully sure I understood your message. Could you please explain it in another way?
Intent: fallback
Confidence: 0.1811
Sentiment: Positive
Emotion: Neutral
Entities: {'service': ['internet', 'connection']}
User: I am very frustrated with the billing problem.
Bot: I'm not fully sure I understood your message. Could you please explain it in another way?
Intent: fallback
Confidence: 0.1775
Sentiment: Negative
Emotion: Frustrated
Entities: {'service': ['billing']}
User: Thank you for your support.
Bot: I'm not fully sure I understood your message. Could you please explain it in another way?
Intent: fallback
Confidence: 0.2305
Sentiment: Positive
Emotion: Neutral
Entities: {}
User: I want to speak with a human agent.
Bot: I'm not 

In [22]:
# interactive chatbot interface

# Chat interface components
chat_input = widgets.Text(
    placeholder="Type your message here...",
    description="You:",
    layout=widgets.Layout(width="75%")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="paper-plane"
)

clear_button = widgets.Button(
    description="Clear Chat",
    button_style="warning"
)

chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid gray",
        padding="10px",
        height="350px",
        overflow="auto"
    )
)

# Store conversations during this notebook session
chat_history = []


def send_message(button):
    message = chat_input.value.strip()

    if not message:
        return

    result = process_message(message)

    with chat_output:
        print("You:", message)
        print("Bot:", result["response"])
        print("Intent:", result["intent"])
        print("Confidence:", result["confidence"])
        print("-" * 45)

    chat_history.append({
        "timestamp": datetime.now().isoformat(),
        "message": message,
        "response": result["response"],
        "intent": result["intent"],
        "confidence": result["confidence"],
        "sentiment": result["sentiment"],
        "emotion": result["emotion"],
        "entities": json.dumps(result["entities"])
    })

    chat_input.value = ""


def clear_chat(button):
    with chat_output:
        clear_output()

    chat_history.clear()


send_button.on_click(send_message)
clear_button.on_click(clear_chat)

display(
    widgets.VBox([
        widgets.HTML("<h3>Dynamic AI Customer Support Chatbot</h3>"),
        chat_output,
        widgets.HBox([chat_input, send_button]),
        clear_button
    ])
)

print("Interactive chatbot interface created.")

Interactive chatbot interface created.


In [24]:
# Save chatbot interactions
def save_interaction_history():
    if not chat_history:
        print("No interactions available to save.")
        return

    history_df = pd.DataFrame(chat_history)

    if HISTORY_PATH.exists():
        existing_df = pd.read_csv(HISTORY_PATH)
        history_df = pd.concat(
            [existing_df, history_df],
            ignore_index=True
        )

    history_df.to_csv(HISTORY_PATH, index=False)

    print("Interaction history saved:", HISTORY_PATH)


# Collect user feedback
def save_user_feedback(feedback):
    if not chat_history:
        print("Send a message before submitting feedback.")
        return

    last_interaction = chat_history[-1]

    feedback_record = {
        "timestamp": datetime.now().isoformat(),
        "message": last_interaction["message"],
        "predicted_intent": last_interaction["intent"],
        "confidence": last_interaction["confidence"],
        "feedback": feedback
    }

    feedback_df = pd.DataFrame([feedback_record])

    if FEEDBACK_PATH.exists():
        feedback_df.to_csv(
            FEEDBACK_PATH,
            mode="a",
            header=False,
            index=False
        )
    else:
        feedback_df.to_csv(FEEDBACK_PATH, index=False)

    print("Feedback saved:", feedback)


# Feedback buttons
positive_feedback_button = widgets.Button(
    description="Helpful",
    button_style="success",
    icon="thumbs-up"
)

negative_feedback_button = widgets.Button(
    description="Not Helpful",
    button_style="danger",
    icon="thumbs-down"
)

positive_feedback_button.on_click(
    lambda button: save_user_feedback("Positive")
)

negative_feedback_button.on_click(
    lambda button: save_user_feedback("Negative")
)

display(
    widgets.HBox([
        positive_feedback_button,
        negative_feedback_button
    ])
)

print("Feedback collection is ready.")

save_interaction_history()

Feedback collection is ready.
Interaction history saved: D:\Dynamic_AI_Chatbot_Project\outputs\chatbot\chatbot_interaction_history.csv


In [25]:
# Final chatbot functionality tests

final_test_messages = [
    "Hello",
    "Can you help me with my internet service?",
    "I am unhappy with the billing service.",
    "Thank you for helping me.",
    "I want to talk to a human agent.",
    "My email is test@example.com and I need payment support."
]

test_results = []

for message in final_test_messages:

    result = process_message(message)

    test_results.append({
        "message": message,
        "intent": result["intent"],
        "confidence": result["confidence"],
        "sentiment": result["sentiment"],
        "emotion": result["emotion"],
        "response": result["response"]
    })

final_test_df = pd.DataFrame(test_results)

display(final_test_df)

print("Total test messages:", len(final_test_df))
print("Empty responses:", final_test_df["response"].isna().sum())
print("Empty intents:", final_test_df["intent"].isna().sum())

,message,intent,confidence,sentiment,emotion,response
0,Hello,fallback,0.3033,Neutral,Neutral,I'm not fully sure I understood your message. ...
1,Can you help me with my internet service?,fallback,0.1992,Positive,Neutral,I'm not fully sure I understood your message. ...
2,I am unhappy with the billing service.,fallback,0.1488,Negative,Happy,I'm not fully sure I understood your message. ...
3,Thank you for helping me.,fallback,0.2149,Positive,Neutral,I'm not fully sure I understood your message. ...
4,I want to talk to a human agent.,fallback,0.3368,Positive,Neutral,I'm not fully sure I understood your message. ...
5,My email is test@example.com and I need paymen...,fallback,0.2211,Positive,Neutral,I'm not fully sure I understood your message. ...


Total test messages: 6
Empty responses: 0
Empty intents: 0


In [26]:
TEST_REPORT_PATH = OUTPUT_DIR / "final_chatbot_test_report.csv"

final_test_df.to_csv(TEST_REPORT_PATH, index=False)

print("Final test report saved successfully.")
print("Location:", TEST_REPORT_PATH)

Final test report saved successfully.
Location: D:\Dynamic_AI_Chatbot_Project\outputs\chatbot\final_chatbot_test_report.csv


In [27]:
required_files = [
    HISTORY_PATH,
    FEEDBACK_PATH,
    TEST_REPORT_PATH
]

print("FINAL OUTPUT VERIFICATION")
print("=" * 50)

for file_path in required_files:
    print(
        file_path.name,
        "| Exists:", file_path.exists()
    )

FINAL OUTPUT VERIFICATION
chatbot_interaction_history.csv | Exists: True
chatbot_feedback.csv | Exists: False
final_chatbot_test_report.csv | Exists: True
